# C3 · Open-vocabulary detection with YOLO-World (12 min)

**Question:** can a detector find objects named in plain English that were never in its fixed class list, and how much does the wording matter?

Flow: closed-set YOLO vs YOLO-World on photos → synonym sensitivity → zero-shot accuracy on COCO128 → a class COCO does not have → save a custom-vocabulary model.
Outputs go to `outputs/c3/`.

In [ ]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# ---- configuration -------------------------------------------------------
CLOSED_WEIGHTS = "yolo11s.pt"           # fixed 80 COCO classes
WORLD_WEIGHTS  = "yolov8s-worldv2.pt"   # open vocabulary
CUSTOM_CLASSES = ["necktie", "beard", "bald head", "suit jacket", "window", "wheel", "traffic sign"]
SYNONYMS = {                            # same concept, different words
    "tie":    ["tie", "necktie", "red tie"],
    "beard":  ["beard", "facial hair", "man with beard"],
    "wheel":  ["wheel", "tire", "bus wheel"],
}
NOVEL_CLASS = "glasses"                 # not a COCO class
CONF = 0.10

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO, YOLOWorld
from vlmkit.common import out_dir, save_json, show_grid, Timer
from vlmkit import data

OUT = out_dir("c3")
samples = data.sample_images()
names, coco = data.coco128()
closed = YOLO(CLOSED_WEIGHTS)
world = YOLOWorld(WORLD_WEIGHTS)

## 1. Closed set vs open vocabulary on the same photos

In [ ]:
def run(model, path, conf=CONF):
    r = model.predict(str(path), conf=conf, verbose=False)[0]
    found = pd.DataFrame({"class": [r.names[int(c)] for c in r.boxes.cls],
                          "conf": r.boxes.conf.cpu().numpy().round(3)})
    return Image.fromarray(r.plot()[:, :, ::-1]), found

world.set_classes(CUSTOM_CLASSES)
panels, titles = [], []
for name, path in samples.items():
    img_c, f_c = run(closed, path, conf=0.25)
    img_w, f_w = run(world, path)
    panels += [img_c, img_w]; titles += [f"{name}: closed set", f"{name}: open vocabulary"]
    print(f"\n{name}\n closed set : {sorted(set(f_c['class']))}\n open vocab : {sorted(set(f_w['class']))}")
show_grid(panels, titles, cols=2, size=4.5, save=OUT / "closed_vs_open.png")

## 2. Wording sensitivity

The same concept, different phrasings: detections and best confidence per phrasing.

In [ ]:
rows = []
for concept, variants in SYNONYMS.items():
    for v in variants:
        world.set_classes([v])
        for name, path in samples.items():
            r = world.predict(str(path), conf=CONF, verbose=False)[0]
            confs = r.boxes.conf.cpu().numpy()
            rows.append({"concept": concept, "phrase": v, "image": name,
                         "detections": len(confs), "best_conf": float(confs.max()) if len(confs) else 0.0})
syn = pd.DataFrame(rows)
syn.to_csv(OUT / "synonyms.csv", index=False)
best = syn.groupby(["concept", "phrase"])["best_conf"].max().unstack(0)
ax = best.plot.bar(figsize=(8, 3.5), rot=30); ax.set_ylabel("best confidence over the photos")
ax.set_title("Same concept, different words, different confidence"); plt.tight_layout()
plt.savefig(OUT / "synonyms.png", dpi=120); plt.show()
syn.pivot_table(index=["concept", "phrase"], columns="image", values="best_conf").round(2)

## 3. Zero-shot accuracy on COCO128

YOLO-World is given the 80 COCO class names as text and compared with a detector trained on COCO.
COCO128 images come from COCO's training set, so read this as a sanity check of the open-vocabulary model, not a fair contest.

In [ ]:
coco_names = [names[i] for i in sorted(names)]
results = []
for label, model in [("YOLO11s (trained on COCO)", YOLO(CLOSED_WEIGHTS)),
                     ("YOLO-World (COCO names as text)", YOLOWorld(WORLD_WEIGHTS))]:
    if isinstance(model, YOLOWorld):
        model.set_classes(coco_names)
    m = model.val(data="coco128.yaml", imgsz=640, batch=8, verbose=False, plots=False)
    results.append({"model": label, "mAP50": round(m.box.map50, 3), "mAP50-95": round(m.box.map, 3),
                    "ms_per_image": round(m.speed["inference"], 1)})
val_df = pd.DataFrame(results); val_df.to_csv(OUT / "coco128_val.csv", index=False)
val_df

## 4. A class COCO does not have

Search all 128 images for the novel class and inspect the most confident boxes.

In [ ]:
world.set_classes([NOVEL_CLASS])
hits = []
for it in coco:
    r = world.predict(str(it["path"]), conf=0.2, verbose=False)[0]
    for box, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        hits.append((float(c), it["path"], box))
hits.sort(key=lambda h: -h[0])
print(f"'{NOVEL_CLASS}': {len(hits)} boxes in {len({h[1] for h in hits})} images (conf >= 0.2)")
crops = []
for c, path, (x1, y1, x2, y2) in hits[:10]:
    im = Image.open(path).convert("RGB")
    pad = 20
    crops.append(im.crop((max(x1 - pad, 0), max(y1 - pad, 0), x2 + pad, y2 + pad)).resize((160, 160)))
if crops:
    show_grid(crops, [f"{h[0]:.2f}" for h in hits[:10]], cols=5, size=2.2, save=OUT / f"novel_{NOVEL_CLASS}.png")
print("Judge by eye: how many of the top 10 are correct? That is precision@10 for a class with no labels.")

## 5. Save a model with your vocabulary baked in

In [ ]:
world.set_classes(CUSTOM_CLASSES)
saved = OUT / "custom_vocab_world.pt"
world.save(str(saved))
reloaded = YOLO(str(saved))
print("reloaded classes:", list(reloaded.names.values()))
save_json({"custom_classes": CUSTOM_CLASSES, "coco128": val_df.to_dict("records"),
           "novel_class": NOVEL_CLASS, "novel_hits": len(hits),
           "synonym_best_conf": best.round(3).to_dict()}, OUT / "summary.json")

## 6. Try it live

Type any comma-separated object names, pick a photo and a confidence threshold, then press **Run**. Tick *compare_closed* to see the fixed-class detector next to it.

In [ ]:
from vlmkit.common import live

choices = {f"sample / {name}": str(p) for name, p in samples.items()}
choices.update({f"coco128 / {it['path'].name}": str(it["path"]) for it in coco[:60]})

def detect_panel(image, classes, conf, compare_closed):
    world.set_classes([c.strip() for c in classes.split(",") if c.strip()])
    img_w, found = run(world, image, conf=conf)
    panels, titles = [img_w], [f"open vocabulary: {classes}"]
    if compare_closed:
        img_c, _ = run(closed, image, conf=0.25)
        panels, titles = [img_c, img_w], ["closed set (80 COCO classes)", titles[0]]
    show_grid(panels, titles, cols=len(panels), size=5.5)
    print(found.groupby("class").conf.agg(["count", "max"]).sort_values("max", ascending=False)
          if len(found) else "nothing found: lower conf or rephrase")

live(detect_panel, image=choices, classes="person, tie, glasses, shoe, bus window",
     conf=(0.01, 0.9, 0.01, 0.1), compare_closed=True)

## Exercises for students

1. Add three classes your project needs but your dataset lacks (for example "ambulance", "debris on road", "person lying on road") and test them on your own frames.
2. In section 2, find a phrasing that doubles the confidence of a weak class.
3. Fill in precision@10 for `NOVEL_CLASS` by eye, then try a second novel class.